# Chapter 09. 파이썬 입출력과 데이터 저장

이 노트북은 [Chapter 09 원본 문서](../doc/Chapter%2009.%20%ED%8C%8C%EC%9D%B4%EC%8D%AC%20%EC%9E%85%EC%B6%9C%EB%A0%A5%EA%B3%BC%20%EB%8D%B0%EC%9D%B4%ED%84%B0%20%EC%A0%80%EC%9E%A5.md)를 실습용으로 변환한 자료입니다.

문자열 포맷팅, 파일 읽기·쓰기, 파일 위치, JSON 저장을 코인 거래 데이터로 연습합니다.

## 1. 실습 환경 준비

파일 실습 결과가 프로젝트 폴더에 남지 않도록 임시 디렉터리를 준비합니다.

In [1]:
from pathlib import Path
import tempfile

work_dir = Path(tempfile.mkdtemp(prefix="chapter09_"))
print("실습 디렉터리:", work_dir)

실습 디렉터리: C:\Users\juyoung\AppData\Local\Temp\chapter09_o1uaoptf


## 2. `print()`, `str()`, `repr()`

`print()`의 `sep`와 `end`, 사람이 읽는 `str()`, 디버깅에 유용한 `repr()`을 확인합니다.

In [2]:
symbol = "BTC"
price = 105_000_000
message = "Hello, world.\n"

print("거래 대상:", symbol, "현재 가격:", price)
print("BTC", "ETH", "XRP", sep=" / ")
print("같은 줄", end=" ")
print("이어지는 출력")
print("str:", str(message))
print("repr:", repr(message))

거래 대상: BTC 현재 가격: 105000000
BTC / ETH / XRP
같은 줄 이어지는 출력
str: Hello, world.

repr: 'Hello, world.\n'


## 3. f-string 포맷팅

f-string의 표현식, 백분율, 천 단위 구분 기호, 정렬과 디버깅 표현을 실습합니다.

In [3]:
symbol = "BTC"
price = 105_000_000
change_rate = 2.35
volume = 1234.5678

print(f"{symbol} 가격: {price:,}원")
print(f"변동률: {change_rate:.2f}%")
print(f"수익률: {(110 - 100) / 100:.2%}")
print(f"거래량: {volume:.2f}")
print(f"{symbol:>4} | {price:>12,d}원")
print(f"{price=}, {change_rate=}")
print(f"{symbol!r}")

BTC 가격: 105,000,000원
변동률: 2.35%
수익률: 10.00%
거래량: 1234.57
 BTC |  105,000,000원
price=105000000, change_rate=2.35
'BTC'


## 4. `str.format()`과 수동 정렬

`str.format()`은 위치·키워드 인자를 포맷 문자열에 넣습니다. `rjust()`, `ljust()`, `center()`, `zfill()`도 함께 확인합니다.

In [4]:
market = {"symbol": "BTC", "price": 105_000_000}
print("{symbol} 가격은 {price:,}원입니다.".format(**market))

for number in range(1, 4):
    print("{0:2d}의 제곱은 {1:3d}입니다.".format(number, number ** 2))

label = "BTC"
print(label.rjust(8, "."))
print(label.ljust(8, "-"))
print(label.center(8, "-"))
print("12".zfill(5))
print("-3.14".zfill(7))

BTC 가격은 105,000,000원입니다.
 1의 제곱은   1입니다.
 2의 제곱은   4입니다.
 3의 제곱은   9입니다.
.....BTC
BTC-----
--BTC---
00012
-003.14


## 5. 파일 쓰기와 `with open()`

`with open()`을 사용하면 파일이 자동으로 닫힙니다. 텍스트 파일은 UTF-8 인코딩을 명시하는 습관을 들입니다.

In [5]:
market_path = work_dir / "market.txt"

with open(market_path, "w", encoding="utf-8") as file:
    file.write("BTC,105000000\n")
    file.write("ETH,3500000\n")

with open(market_path, "a", encoding="utf-8") as file:
    written = file.write("XRP,800\n")

print("추가한 문자 수:", written)
print("파일 존재 여부:", market_path.exists())

추가한 문자 수: 8
파일 존재 여부: True


## 6. `read()`, `readline()`, 파일 반복

같은 텍스트 파일을 전체 읽기, 한 줄 읽기, 파일 객체 반복 방식으로 확인합니다.

In [6]:
with open(market_path, encoding="utf-8") as file:
    content = file.read()
print("전체 내용:\n", content)

with open(market_path, encoding="utf-8") as file:
    first_line = file.readline()
    second_line = file.readline()
print("첫 줄 repr:", repr(first_line))
print("둘째 줄 repr:", repr(second_line))

with open(market_path, encoding="utf-8") as file:
    symbols = [line.strip().split(",")[0] for line in file]
print("심볼 목록:", symbols)

전체 내용:
 BTC,105000000
ETH,3500000
XRP,800

첫 줄 repr: 'BTC,105000000\n'
둘째 줄 repr: 'ETH,3500000\n'
심볼 목록: ['BTC', 'ETH', 'XRP']


## 7. 파일 위치와 바이너리 모드

`tell()`은 현재 위치를 반환하고 `seek()`은 위치를 이동합니다. 텍스트가 아닌 데이터는 바이너리 모드로 처리합니다.

In [7]:
with open(market_path, encoding="utf-8") as file:
    print("시작 위치:", file.tell())
    print("앞 세 글자:", file.read(3))
    print("현재 위치:", file.tell())
    file.seek(0)
    print("처음부터 두 글자:", file.read(2))

binary_path = work_dir / "data.bin"
with open(binary_path, "wb") as file:
    file.write(b"0123456789")
with open(binary_path, "rb") as file:
    file.seek(5)
    print("바이너리 6번째 바이트:", file.read(1))

시작 위치: 0
앞 세 글자: BTC
현재 위치: 3
처음부터 두 글자: BT
바이너리 6번째 바이트: b'5'


## 8. JSON 직렬화와 역직렬화

`json.dumps()`와 `json.loads()`는 문자열을 변환하고, `json.dump()`와 `json.load()`는 파일과 직접 연결합니다.

In [8]:
import json

markets = {
    "BTC": {"price": 105_000_000, "change_rate": 2.35},
    "ETH": {"price": 3_500_000, "change_rate": -1.2},
}

json_text = json.dumps(markets, ensure_ascii=False, indent=2)
print(json_text)
restored = json.loads(json_text)
print("복원한 BTC 가격:", restored["BTC"]["price"])

json_path = work_dir / "markets.json"
with open(json_path, "w", encoding="utf-8") as file:
    json.dump(markets, file, ensure_ascii=False, indent=2)
with open(json_path, encoding="utf-8") as file:
    loaded = json.load(file)
print("파일에서 읽은 ETH 가격:", loaded["ETH"]["price"])

{
  "BTC": {
    "price": 105000000,
    "change_rate": 2.35
  },
  "ETH": {
    "price": 3500000,
    "change_rate": -1.2
  }
}
복원한 BTC 가격: 105000000
파일에서 읽은 ETH 가격: 3500000


## 9. 거래 기록 저장 실습

f-string으로 사람이 읽는 거래 메시지를 만들고, 같은 거래 데이터를 JSON 파일로 저장합니다.

In [9]:
trade = {
    "symbol": "BTC",
    "side": "buy",
    "price": 105_000_000,
    "quantity": 0.001,
    "fee_rate": 0.0004,
}

gross_amount = trade["price"] * trade["quantity"]
fee = gross_amount * trade["fee_rate"]
print(
    f"{trade['symbol']} {trade['side']} | "
    f"거래 금액: {gross_amount:,.0f}원 | 수수료: {fee:,.0f}원"
)

trade_path = work_dir / "trade.json"
with open(trade_path, "w", encoding="utf-8") as file:
    json.dump(trade, file, ensure_ascii=False, indent=2)
print("거래 기록 저장:", trade_path)

BTC buy | 거래 금액: 105,000원 | 수수료: 42원
거래 기록 저장: C:\Users\juyoung\AppData\Local\Temp\chapter09_o1uaoptf\trade.json


## 10. 변환 결과 검증

원본 문서, 생성된 파일, JSON 데이터와 계산 결과가 정상인지 확인합니다.

In [10]:
workspace_dir = Path.cwd()
while workspace_dir != workspace_dir.parent and not (workspace_dir / "doc").exists():
    workspace_dir = workspace_dir.parent
source_path = workspace_dir / "doc" / "Chapter 09. 파이썬 입출력과 데이터 저장.md"
assert source_path.exists(), source_path
assert market_path.exists()
assert json_path.exists() and trade_path.exists()
assert loaded["BTC"]["price"] == 105_000_000
assert round(fee, 2) == 42.0
print("Chapter 09 실습 검증 통과")
print("원본 문서:", source_path)
print("생성 파일:", sorted(path.name for path in work_dir.iterdir()))

Chapter 09 실습 검증 통과
원본 문서: c:\Users\juyoung\OneDrive\바탕 화면\python\python-edu-coin-trading\doc\Chapter 09. 파이썬 입출력과 데이터 저장.md
생성 파일: ['data.bin', 'market.txt', 'markets.json', 'trade.json']


## 실습 과제

1. 여러 코인의 심볼, 가격, 변동률을 f-string 표로 출력하세요.
2. `market.txt` 파일에 가격을 한 줄씩 저장하고 다시 읽으세요.
3. `read()`, `readline()`, 파일 반복 방식의 차이를 비교하세요.
4. 거래 기록을 JSON으로 저장하고 다시 불러와 총액과 수수료를 계산하세요.
5. `ensure_ascii=False`와 `indent=2`를 바꿔 JSON 결과를 비교하세요.